# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

###Lane 2 — Refresh / Content Opportunity Scoring.

I'm picking this one because it maps onto a decision someone will actually act on, not just an interesting pattern. "Which pages does the editor open first" is unambiguous, has a natural cost asymmetry (missing a real decliner costs more than reviewing a fine page), and gives a clean baseline-to-beat built into the lane itself — the starter pipeline already reports precision@50 going from ~0.24 (rule) to ~0.74 (model), so I have an honest benchmark for whether the extra modeling complexity earns its keep rather than just assuming it does. It also forces the leakage discipline early (the trend_direction/trend_pct trap), which is exactly the kind of mistake that quietly invalidates a project, so getting it right from the framing stage saves rework later.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Here's the framing for Lane 2, laid out against the three prompts:

**Decision it improves:** Which content pages should be reviewed first out of a large inventory, and what should the reviewer do with each one — refresh, expand, protect, prune, or monitor. Right now that's either scanned manually or ignored; the model turns it into a ranked queue.

**Who acts on it:** A content editor or SEO lead with limited review hours per week. They open the top of the queue, read the reason code attached to each page, and take the suggested action (or override it — the queue is a recommendation, not an automatic decision).

**Cost of a wrong call — and the two directions aren't symmetric:**
- **False positive** — a page gets flagged and reviewed, but it was actually fine. Cost: wasted editor time, the cheaper mistake.
- **False negative** — a genuinely declining page never surfaces in the queue. Cost: it keeps losing traffic silently until someone notices some other way, which is more expensive and harder to catch.

Because a missed decliner costs more than a wasted review, this pushes toward optimizing recall on the "real decline" cases even at some precision cost — worth deciding explicitly rather than defaulting to whatever threshold falls out of the model. It also means precision@K alone won't tell the whole story; a recall check on the top-K against known decliners is worth tracking alongside it.

One more note tying back to `flyrank-data`: since your label chain runs through `trend_direction`/`trend_pct`, and the cost asymmetry above pushes you toward wanting *early* signal (catching decline before it's fully visible), it's worth deciding now whether your target window is "declining in the trailing 90 days" (what's already in the label) or something more predictive — because those imply different features and different claims about what the model is actually doing.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
!git clone https://github.com/Syed8855/FlyRank.git
%cd FlyRank

Cloning into 'FlyRank'...
remote: Enumerating objects: 99, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 99 (delta 21), reused 80 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (99/99), 1.84 MiB | 11.82 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/FlyRank/FlyRank


In [4]:
import pandas as pd

csv_path = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} rows x {len(df.columns)} cols\n")

# --- Quick look: Lane 2 numbers -----------------------------------------
visible = df[df["avg_position"] > 0]                     # avg_position==0 means "no data"
declining = visible[visible["trend_direction"] == "down"]

pct_declining = len(declining) / len(visible) * 100
page1_decay = declining[declining["position_tier"] == "page_1"]
pct_page1 = len(page1_decay) / len(visible) * 100
stale = declining[declining["freshness_tier"].isin(["91-180", "181+"])]
pct_stale = len(stale) / len(visible) * 100

print(f"Declining AND visible: {len(declining)} of {len(visible)} ({pct_declining:.1f}%)")
print(f"Page-one decay risk: {len(page1_decay)} ({pct_page1:.1f}% of visible)")
print(f"Declining AND stale (91+ days): {len(stale)} ({pct_stale:.1f}% of visible)")


Loaded 30000 rows x 44 cols

Declining AND visible: 16254 of 28795 (56.4%)
Page-one decay risk: 6730 (23.4% of visible)
Declining AND stale (91+ days): 5684 (19.7% of visible)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

###What the work CAN say:

* Observed — "In this dataset, 56.4% of visible pages are trending down, and 23.4% are currently ranking on page one while declining." This is a description of the sample, nothing more.
* Associated / directional — "Pages that are both declining and stale (no update in 91+ days) make up a distinct, sizeable slice of the inventory (19.7%), suggesting staleness and decline travel together." This is a measured pattern in one snapshot — not proof one causes the other.
* Decision-support (once the model is validated out-of-sample) — "The model ranks pages by review priority at precision@K of X, beating the hand-rule baseline of ~0.24; these pages look worth an editor's attention first." This is the strongest claim the project earns, and only after a real train/test split with grouped holdout by client_id.

###What the work will NEVER say:

* Anything with "proves," "causes," or "will increase" traffic/clicks/rankings if a page is refreshed. This is cross-sectional, observational data — no intervention was run, so nothing here supports "doing X produces Y."
* "We predicted Google's algorithm." Nobody modeled the algorithm; the model ranked outcomes within one anonymized client portfolio.
Any confident number pulled from a tiny bucket without stating n (e.g. "declining with above-median demand: 16.2%, n=4,663" — always paired, never bare).
* Accuracy or precision quoted without its base rate next to it — 56.4% of visible pages already decline, so a model that just flags "everything" would look deceptively decent without that context stated up front.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.